# 03 — Otimização

## Objetivo

Um tuning curto melhora o Gradient Boosting de forma útil para a aula?

Variamos quatro hiperparâmetros em 20 trials. min_samples_leaf fica fixo em 35,
decisão já validada na prova técnica.

In [1]:
from pathlib import Path
import sys

ponto_atual = Path.cwd().resolve()
RAIZ = next(
    caminho for caminho in (ponto_atual, *ponto_atual.parents)
    if (caminho / "data" / "raw" / "UCI_Credit_Card.csv").exists()
)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


import json
import time
import optuna
import pandas as pd
from sklearn.metrics import average_precision_score

from src.auxiliares import (
    PARAMETROS_REFERENCIA,
    avaliar_probabilidades,
    carregar_base_preparada,
    criar_modelo_gradiente,
    separar_dados,
)
from src.visual_utils import (
    grafico_historico_optuna,
    grafico_importancia_hiperparametros,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)
dados = carregar_base_preparada(RAIZ)
X_treino, X_validacao, X_teste, y_treino, y_validacao, y_teste = separar_dados(dados)

## Qual é o ponto de partida?

In [2]:
modelo_inicial = criar_modelo_gradiente()
modelo_inicial.fit(X_treino, y_treino)
probabilidades_iniciais = modelo_inicial.predict_proba(X_validacao)[:, 1]
resultado_inicial = avaliar_probabilidades(
    "Gradient Boosting inicial",
    y_validacao,
    probabilidades_iniciais,
)
pd.DataFrame([resultado_inicial])

,modelo,limiar,precision,recall,f1,pr_auc,vn,fp,fn,vp,tempo_treino_s
0,Gradient Boosting inicial,0.5,0.693314,0.359457,0.473449,0.54258,4462,211,850,477,NaN


## Que combinação o Optuna deve avaliar?

In [3]:
def objetivo(trial):
    parametros = {
        "n_estimators": trial.suggest_int("n_estimators", 80, 220, step=20),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 4),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0, step=0.1),
        "min_samples_leaf": 35,
    }
    modelo = criar_modelo_gradiente(parametros)
    modelo.fit(X_treino, y_treino)
    probabilidades = modelo.predict_proba(X_validacao)[:, 1]
    return average_precision_score(y_validacao, probabilidades)

## Como executar uma busca controlada?

In [4]:
estudo = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
estudo.enqueue_trial({
    "n_estimators": 220,
    "learning_rate": 0.0303,
    "max_depth": 4,
    "subsample": 0.9,
})
inicio = time.perf_counter()
estudo.optimize(objetivo, n_trials=20, show_progress_bar=False)
tempo_busca = time.perf_counter() - inicio
print(f"Melhor AP / PR-AUC: {estudo.best_value:.4f}")
print(f"Tempo da busca: {tempo_busca:.1f} s")

Melhor AP / PR-AUC: 0.5529
Tempo da busca: 302.5 s


## A busca convergiu e quais parâmetros importaram?

In [5]:
historico = estudo.trials_dataframe(attrs=("number", "value"))
historico["melhor_acumulado"] = historico["value"].cummax()
importancias = optuna.importance.get_param_importances(estudo)

fig = grafico_historico_optuna(historico)
fig.show()

fig = grafico_importancia_hiperparametros(importancias)
fig.show()

## O modelo otimizado melhora a referência?

In [6]:
parametros_otimizados = {
    **estudo.best_params,
    "min_samples_leaf": 35,
}
modelo_otimizado = criar_modelo_gradiente(parametros_otimizados)
modelo_otimizado.fit(X_treino, y_treino)
probabilidades_otimizadas = modelo_otimizado.predict_proba(X_validacao)[:, 1]
resultado_otimizado = avaliar_probabilidades(
    "Gradient Boosting otimizado",
    y_validacao,
    probabilidades_otimizadas,
)
pd.DataFrame([resultado_inicial, resultado_otimizado])

,modelo,limiar,precision,recall,f1,pr_auc,vn,fp,fn,vp,tempo_treino_s
0,Gradient Boosting inicial,0.5,0.693314,0.359457,0.473449,0.542580,4462,211,850,477,NaN
1,Gradient Boosting otimizado,0.5,0.682720,0.363225,0.474176,0.552929,4449,224,845,482,NaN


In [7]:
caminho_parametros = RAIZ / "models" / "parametros_gradient_boosting.json"
caminho_parametros.parent.mkdir(parents=True, exist_ok=True)
caminho_parametros.write_text(
    json.dumps(parametros_otimizados, indent=2),
    encoding="utf-8",
)
print(f"Parâmetros salvos em: {caminho_parametros}")
parametros_otimizados

Parâmetros salvos em: C:\GitHub\ALURA-TESTE\classificacao-avancando-classificadores\models\parametros_gradient_boosting.json


{'n_estimators': 220,
 'learning_rate': 0.0303,
 'max_depth': 4,
 'subsample': 0.9,
 'min_samples_leaf': 35}

## Resultado

O ganho esperado é modesto, próximo ao observado na prova técnica. Isso é
pedagogicamente útil: tuning organiza a busca, mas não garante salto grande.
O teste continua intocado.